# Tyre Degradation — Lap Time Modeling (Ridge → Random Forest → Pace Delta)

Predicting F1 race lap time (seconds) from tire age, compound, a race-phase bucket, track
status, driver, and circuit, on 2021–2025 FastF1 race laps. Data comes from the project
pipeline (`data_pull.py` → `split.py`): deduplicated `IsAccurate` laps, split into
train/dev/test by whole races (per-circuit, per-year) so laps from one race never straddle
splits.

Structure: Ridge baselines and ablations on absolute lap time (§4), a Random Forest on the
same features (§5), then a change of target to the within-race pace delta (§6) — the
follow-up direction from `notes/degradation_experiment_report.md`, which showed the absolute
target is so dominated by circuit identity that the tyre signal only becomes readable once
the circuit baseline is subtracted out of the target.

**Cleanup notes (2026-07-17).** This notebook was reorganized from an out-of-order set of
experiment cells into a linear run. Bugs fixed during cleanup, without changing any modeling
logic:

1. **Stale-variable evaluation bugs.** Dev matrices were assembled from whatever
   `numeric_features` / `categorical_features` were left in kernel memory — once producing a
   dev matrix with `LapNumber` and all Driver dummies zeroed (RF dev R² collapsed to 0.05),
   once accidentally using the *train* matrix as "dev" (KeyError). Train and dev matrices are
   now built by one shared `build_matrix()` function so their feature recipes cannot diverge.
2. **Model-variable rebinding.** The TyreLife coefficient was once printed from `model` after
   that name had been rebound to a different fit. Every model now has a distinct name
   (`ridge_full`, `ridge_nodriver`, `ridge_fuel`, `rf_fuel`, `ridge_delta`, `rf_delta`) and
   coefficients are read from the named model.
3. **`track_status_bucket` now handles SC/VSC codes** ('4', '6', '7' → yellow). Behavior on
   the current data is identical (those codes don't survive the `IsAccurate` filter), so no
   result changes — it just won't silently mislabel them if the upstream filter ever changes.

## 1. Setup / Imports

Everything the notebook needs, up front: pandas for the data, Ridge and RandomForest as the
two model families, and MSE/R² as the dev-set metrics used throughout.

In [1]:
# imports for data handling, the two model families, and evaluation metrics
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

## 2. Data Loading

`split.py` splits by **whole race** (a circuit-year pair): if half a race's laps were in train
and the other half in dev, race-specific conditions (weather, safety cars, track evolution)
would leak across and inflate dev scores. Dev is therefore a separate file of *entire held-out
races*, and every dev matrix below must be built from `dev_df` with exactly the same feature
recipe as train — never reused train-side variables. Sizes: train 62,748 laps / dev 22,183
laps (post-dedup).

In [2]:
# load the train/dev splits produced by split.py (deduplicated laps, split by whole race)
train_df = pd.read_csv('../data/train.csv', low_memory=False)
dev_df = pd.read_csv('../data/dev.csv', low_memory=False)
print(train_df.shape, dev_df.shape)

(62748, 35) (22183, 35)


## 3. Feature Engineering

Features and why they're in:

- **`TyreLife`** (numeric) — laps on the current tire set; the variable of interest.
  Degradation should show up as a *positive* lap-time slope.
- **`Compound`** (one-hot) — soft/medium/hard/inter/wet have different base pace, so without
  it soft-tire laps just look "fast".
- **`WeatherBucket`** (numeric 1–4) — which quarter of the race the lap falls in. A coarse
  conditions/track-evolution proxy (note: the pipeline never merged real weather data, so this
  is race progress, not measured weather).
- **`TrackStatusBucket`** — FastF1 encodes track status as concatenated digit codes
  (1 clear, 2 yellow, 4 SC, 5 red, 6/7 VSC; a lap can carry combos like '12' or '21'). Yellow
  laps are slow for non-tire reasons, so they need their own bucket. Current data contains
  only {1, 2, 12, 21}.
- **`Driver`** (one-hot) — absorbs car+driver pace (seconds/lap between teams). Identity
  feature: powerful but tells us nothing about tires.
- **`location`** (one-hot) — circuit baseline; lap times range ~70 s to ~105 s across
  circuits, so absolute lap-time prediction is meaningless without it. Also an identity
  feature.

`build_matrix()` is the single place feature matrices get built. Passing
`train_columns=` reindexes a dev matrix to the train columns (unseen dummy categories become
0), guaranteeing train and dev always share one recipe — this is the fix for the two
stale-variable bugs described at the top.

In [3]:
# bucket FastF1's concatenated flag codes into green / yellow / red
def track_status_bucket(status):
    status = str(status)
    if '5' in status:
        return 'red'
    elif any(code in status for code in ('2', '4', '6', '7')):
        return 'yellow'
    else:
        return 'green'

train_df['TrackStatusBucket'] = train_df['TrackStatus'].apply(track_status_bucket)
dev_df['TrackStatusBucket'] = dev_df['TrackStatus'].apply(track_status_bucket)
print(train_df['TrackStatusBucket'].value_counts())
print(dev_df['TrackStatusBucket'].value_counts())

TrackStatusBucket
green     60670
yellow     2078
Name: count, dtype: int64
TrackStatusBucket
green     21693
yellow      490
Name: count, dtype: int64


In [4]:
# one shared builder for every feature matrix: numeric columns + one-hot categoricals,
# aligned to the train columns when building a dev matrix. `target` names the column
# returned as y, so the same builder serves the absolute and pace-delta experiments.
def build_matrix(df, numeric_cols, categorical_cols, train_columns=None, target='LapTime'):
    X = pd.concat([df[numeric_cols], pd.get_dummies(df[categorical_cols])], axis=1)
    if train_columns is not None:
        X = X.reindex(columns=train_columns, fill_value=0)
    X = X.dropna()
    y = df[target].loc[X.index]
    return X, y

CATEGORICAL_COLS = ['Compound', 'Driver', 'TrackStatusBucket', 'location']

## 4. Ridge Regression (Baseline)

Three linear experiments, each asking one question:

- **4a — baseline**: how well do these features predict lap time at all? (Answer: dev
  R² ≈ 0.91, RMSE ≈ 3.5 s.)
- **4b — ablation, no Driver**: how much of that comes from identity features? Dropping
  Driver barely moves the score (≈ 0.90); side diagnostics showed dropping Driver *and*
  location collapses R² to ≈ 0.02 — the model is mostly a circuit-baseline lookup, not a
  degradation model.
- **4c — fuel proxy**: within a stint, tire age and fuel load rise together, so the raw
  TyreLife slope is confounded by the car getting lighter/faster. `LapNumber` is added as a
  fuel-load proxy to separate the two.

The TyreLife coefficients at the end are the headline: negative in 4a/4b (physically
backwards), flipping to slightly positive (≈ +0.007 s/lap) once fuel is proxied in 4c — the
right sign, but an order of magnitude below realistic degradation, which motivates the
non-linear model in section 5.

In [5]:
# 4a: build train/dev matrices for the baseline ridge (no fuel proxy)
X_train_full, y_train_full = build_matrix(
    train_df, ['TyreLife', 'WeatherBucket'], CATEGORICAL_COLS)
X_dev_full, y_dev_full = build_matrix(
    dev_df, ['TyreLife', 'WeatherBucket'], CATEGORICAL_COLS, train_columns=X_train_full.columns)
print(X_train_full.shape, X_dev_full.shape)

(62591, 72) (21526, 72)


In [6]:
# 4a: fit the baseline ridge and score it on dev
ridge_full = Ridge(alpha=1.0)
ridge_full.fit(X_train_full, y_train_full)
pred_dev_full = ridge_full.predict(X_dev_full)
mse_ridge_full = mean_squared_error(y_dev_full, pred_dev_full)
r2_ridge_full = r2_score(y_dev_full, pred_dev_full)
print(f"ridge baseline   dev MSE={mse_ridge_full:.3f}  R2={r2_ridge_full:.4f}")

ridge baseline   dev MSE=12.310  R2=0.9102


In [7]:
# 4b: same recipe minus Driver - how much does knowing who is driving contribute?
X_train_nodriver, y_train_nodriver = build_matrix(
    train_df, ['TyreLife', 'WeatherBucket'], ['Compound', 'TrackStatusBucket', 'location'])
X_dev_nodriver, y_dev_nodriver = build_matrix(
    dev_df, ['TyreLife', 'WeatherBucket'], ['Compound', 'TrackStatusBucket', 'location'],
    train_columns=X_train_nodriver.columns)
ridge_nodriver = Ridge(alpha=1.0)
ridge_nodriver.fit(X_train_nodriver, y_train_nodriver)
pred_dev_nodriver = ridge_nodriver.predict(X_dev_nodriver)
mse_ridge_nodriver = mean_squared_error(y_dev_nodriver, pred_dev_nodriver)
r2_ridge_nodriver = r2_score(y_dev_nodriver, pred_dev_nodriver)
print(f"ridge no-driver  dev MSE={mse_ridge_nodriver:.3f}  R2={r2_ridge_nodriver:.4f}")

ridge no-driver  dev MSE=13.488  R2=0.9016


In [8]:
# 4c: add LapNumber as a fuel-load proxy (cars start heavy and speed up as fuel burns)
X_train_fuel, y_train_fuel = build_matrix(
    train_df, ['TyreLife', 'WeatherBucket', 'LapNumber'], CATEGORICAL_COLS)
X_dev_fuel, y_dev_fuel = build_matrix(
    dev_df, ['TyreLife', 'WeatherBucket', 'LapNumber'], CATEGORICAL_COLS,
    train_columns=X_train_fuel.columns)

ridge_fuel = Ridge(alpha=1.0)
ridge_fuel.fit(X_train_fuel, y_train_fuel)
pred_dev_fuel = ridge_fuel.predict(X_dev_fuel)
mse_ridge_fuel = mean_squared_error(y_dev_fuel, pred_dev_fuel)
r2_ridge_fuel = r2_score(y_dev_fuel, pred_dev_fuel)
print(f"ridge fuel-proxy dev MSE={mse_ridge_fuel:.3f}  R2={r2_ridge_fuel:.4f}")

ridge fuel-proxy dev MSE=12.294  R2=0.9103


In [9]:
# read the TyreLife slope from each ridge model by name (no shared `model` variable)
for name, mdl, cols in [('baseline', ridge_full, X_train_full.columns),
                        ('no driver', ridge_nodriver, X_train_nodriver.columns),
                        ('fuel proxy', ridge_fuel, X_train_fuel.columns)]:
    print(f"{name:11s} TyreLife coef: {mdl.coef_[cols.get_loc('TyreLife')]:+.4f} s/lap")
print(f"{'fuel proxy':11s} LapNumber coef: {ridge_fuel.coef_[X_train_fuel.columns.get_loc('LapNumber')]:+.4f} s/lap")

baseline    TyreLife coef: -0.0045 s/lap
no driver   TyreLife coef: -0.0064 s/lap
fuel proxy  TyreLife coef: +0.0067 s/lap
fuel proxy  LapNumber coef: -0.0531 s/lap


## 5. Random Forest

Why RF: the linear model can't represent tire behavior that is non-linear in TyreLife
(warm-up phase, plateau, cliff) or that differs by compound and circuit — a forest can learn
those interactions without hand-specifying them. It is trained on **exactly the same feature
matrix as the fuel-proxy ridge** (`X_train_fuel`) and evaluated on the matching dev matrix
built by the same `build_matrix()` call, which is the fix for the earlier broken evaluations.

Hyperparameters are the last ones tried in the original experiments (100 full-depth trees,
`min_samples_leaf=2`; an earlier `max_depth=10` run underfit badly — train R² 0.75 — because
depth-limited trees can't isolate 28 one-hot circuit columns). Result to watch: the
**train-vs-dev gap** (≈ 0.99 vs ≈ 0.86) — this forest overfits and lands *below* the ridge
baseline on dev, so as configured it does not yet deliver the hoped-for improvement.

In [10]:
# fit the forest on the same features as the fuel-proxy ridge
# (n_jobs=-1 only parallelizes training; results are fixed by random_state)
rf_fuel = RandomForestRegressor(n_estimators=100, min_samples_leaf=2, random_state=42, n_jobs=-1)
rf_fuel.fit(X_train_fuel, y_train_fuel)

,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"m

In [11]:
# score the forest on train and dev - the gap between them measures overfitting
pred_train_rf = rf_fuel.predict(X_train_fuel)
pred_dev_rf = rf_fuel.predict(X_dev_fuel)
r2_rf_train = r2_score(y_train_fuel, pred_train_rf)
mse_rf_dev = mean_squared_error(y_dev_fuel, pred_dev_rf)
r2_rf_dev = r2_score(y_dev_fuel, pred_dev_rf)
print(f"random forest  train R2={r2_rf_train:.4f}")
print(f"random forest  dev   MSE={mse_rf_dev:.3f}  R2={r2_rf_dev:.4f}")

random forest  train R2=0.9935
random forest  dev   MSE=19.031  R2=0.8612


In [12]:
# which features the forest actually leans on
rf_importances = pd.Series(rf_fuel.feature_importances_, index=X_train_fuel.columns)
rf_importances = rf_importances.sort_values(ascending=False)
print(rf_importances.head(10).to_string())
print(f"\nTyreLife importance: {rf_importances['TyreLife']:.4f}")
print(f"LapNumber importance: {rf_importances['LapNumber']:.4f}")

location_Spielberg            0.180311
location_Baku                 0.123307
location_Spa-Francorchamps    0.103798
location_Zandvoort            0.073956
location_Austin               0.057857
location_São Paulo            0.044145
location_Monaco               0.043876
Compound_INTERMEDIATE         0.031808
location_Montréal             0.030671
location_Mexico City          0.030408

TyreLife importance: 0.0080
LapNumber importance: 0.0228


## 6. Pace-Delta Target

Every model above spends its capacity predicting the circuit baseline — that is where ~90% of
the R² comes from, and the tyre features are left fighting over scraps. The follow-up
experiment (see `notes/degradation_experiment_report.md`) showed the fix is to change the
*target*, not the features: predict each lap's gap to that driver's best green-flag lap of the
same race (`LapTimeDelta`), so the circuit/car baseline is subtracted out of the target
instead of being the main thing predicted.

The delta R² is not comparable to the absolute R² above — it measures explained
*within-race* pace variation, the part of the problem the absolute models were barely touching.
The number to watch is the TyreLife slope, which should grow once the baseline stops
dominating.

In [13]:
# per-driver personal best = fastest green-flag lap of that race; delta = lap time above it
green_laps_train = train_df[train_df['TrackStatusBucket'] == 'green']
train_df['PersonalBest'] = green_laps_train.groupby(['Driver', 'year', 'location'])['LapTime'].transform('min')
train_df['PersonalBest'] = train_df.groupby(['Driver', 'year', 'location'])['PersonalBest'].transform('min')
train_df['LapTimeDelta'] = train_df['LapTime'] - train_df['PersonalBest']

green_laps_dev = dev_df[dev_df['TrackStatusBucket'] == 'green']
dev_df['PersonalBest'] = green_laps_dev.groupby(['Driver', 'year', 'location'])['LapTime'].transform('min')
dev_df['PersonalBest'] = dev_df.groupby(['Driver', 'year', 'location'])['PersonalBest'].transform('min')
dev_df['LapTimeDelta'] = dev_df['LapTime'] - dev_df['PersonalBest']

# drivers with no green-flag lap in a race have no baseline - drop those laps
train_delta_df = train_df[train_df['LapTimeDelta'].notna()]
dev_delta_df = dev_df[dev_df['LapTimeDelta'].notna()]
print(f"train {len(train_delta_df)} laps, dev {len(dev_delta_df)} laps with a delta target")

train 62746 laps, dev 22183 laps with a delta target


In [14]:
# same fuel-proxy feature set as 4c, but predicting the pace delta
X_train_delta, y_train_delta = build_matrix(
    train_delta_df, ['TyreLife', 'WeatherBucket', 'LapNumber'], CATEGORICAL_COLS,
    target='LapTimeDelta')
X_dev_delta, y_dev_delta = build_matrix(
    dev_delta_df, ['TyreLife', 'WeatherBucket', 'LapNumber'], CATEGORICAL_COLS,
    train_columns=X_train_delta.columns, target='LapTimeDelta')

ridge_delta = Ridge(alpha=1.0)
ridge_delta.fit(X_train_delta, y_train_delta)
pred_dev_delta = ridge_delta.predict(X_dev_delta)
mse_ridge_delta = mean_squared_error(y_dev_delta, pred_dev_delta)
r2_ridge_delta = r2_score(y_dev_delta, pred_dev_delta)
print(f"ridge delta      dev MSE={mse_ridge_delta:.3f}  R2={r2_ridge_delta:.4f}")
print(f"TyreLife coef:  {ridge_delta.coef_[X_train_delta.columns.get_loc('TyreLife')]:+.4f} s/lap")
print(f"LapNumber coef: {ridge_delta.coef_[X_train_delta.columns.get_loc('LapNumber')]:+.4f} s/lap")

ridge delta      dev MSE=4.451  R2=0.2715
TyreLife coef:  +0.0021 s/lap
LapNumber coef: -0.0479 s/lap


In [15]:
# random forest on the delta target - same config as section 5
rf_delta = RandomForestRegressor(n_estimators=100, min_samples_leaf=2, random_state=42, n_jobs=-1)
rf_delta.fit(X_train_delta, y_train_delta)
pred_dev_rf_delta = rf_delta.predict(X_dev_delta)
mse_rf_delta = mean_squared_error(y_dev_delta, pred_dev_rf_delta)
r2_rf_delta = r2_score(y_dev_delta, pred_dev_rf_delta)
print(f"rf delta         dev MSE={mse_rf_delta:.3f}  R2={r2_rf_delta:.4f}")
rf_delta_importances = pd.Series(rf_delta.feature_importances_, index=X_train_delta.columns).sort_values(ascending=False)
print(rf_delta_importances.head(10).to_string())
print(f"\nTyreLife importance: {rf_delta_importances['TyreLife']:.4f} (rank {list(rf_delta_importances.index).index('TyreLife') + 1})")

rf delta         dev MSE=5.287  R2=0.1347
Compound_INTERMEDIATE    0.206265
Compound_WET             0.148613
LapNumber                0.139340
location_São Paulo       0.094055
TyreLife                 0.071218
location_Istanbul        0.065159
location_Suzuka          0.060688
WeatherBucket            0.054404
location_Silverstone     0.015265
location_Monaco          0.012780

TyreLife importance: 0.0712 (rank 5)


## 7. Comparison / Conclusion

All dev-set results side by side (the two targets are scored on different scales, so compare
rows only within a target). What they mean:

- **Circuit (and driver) identity dominates the absolute target.** All absolute models get
  ~90% of their R² from knowing which circuit the lap is on; the tire features contribute only
  marginally (4b, and the R²→0.02 collapse when location is also removed).
- **The TyreLife signal is real but tiny and confounded on the absolute target.** Linear
  regression recovers the right sign only after proxying fuel load, at ≈ +0.007 s/lap — ~10×
  smaller than realistic degradation. Non-linearity, per-compound rates, and pit-stop
  survivorship (heavily degraded laps get pitted away before they happen) are all invisible to
  a global linear slope.
- **The forest, as configured, overfits the absolute target instead of helping** (train ≈ 0.99
  vs dev ≈ 0.86, below the ridge baseline's ≈ 0.91).
- **The delta target makes the tyre signal more visible, but these features aren't enough to
  read it linearly.** Ridge explains ~27% of within-race pace variation — the part of the
  problem the absolute models were barely touching — yet its TyreLife slope stays tiny
  (+0.002 s/lap): with only these columns, LapNumber still absorbs most of the within-stint
  trend. The forest tells a different story: TyreLife's importance jumps from 0.008 (absolute)
  to 0.071 (rank 5) on the delta, and compound/weather features move to the top — the model is
  finally reasoning about pace decay instead of circuit identity (though it overfits and
  trails ridge on R² here too). The follow-up experiment with enriched features
  (`notes/degradation_experiment_report.md`) confirms the direction: traffic, stint and real
  weather context push the linear slope to +0.026 s/lap and produce physically sensible
  per-compound degradation curves.
- **Next steps:** tune the forests (leaf size, depth, feature subsampling), fold the enriched
  features (traffic gap, stint context, merged weather) into the pipeline proper, and model
  per-compound degradation explicitly.

In [16]:
# collect the dev-set results for every model in one table
results = pd.DataFrame({
    'model': ['ridge baseline', 'ridge no driver', 'ridge + fuel proxy', 'random forest',
              'ridge (delta)', 'random forest (delta)'],
    'target': ['LapTime'] * 4 + ['LapTimeDelta'] * 2,
    'dev MSE': [mse_ridge_full, mse_ridge_nodriver, mse_ridge_fuel, mse_rf_dev,
                mse_ridge_delta, mse_rf_delta],
    'dev R2': [r2_ridge_full, r2_ridge_nodriver, r2_ridge_fuel, r2_rf_dev,
               r2_ridge_delta, r2_rf_delta],
}).set_index('model')
results.round(4)

,target,dev MSE,dev R2
model,,,
ridge baseline,LapTime,12.3103,0.9102
ridge no driver,LapTime,13.4879,0.9016
ridge + fuel proxy,LapTime,12.2944,0.9103
random forest,LapTime,19.0315,0.8612
ridge (delta),LapTimeDelta,4.4510,0.2715
random forest (delta),LapTimeDelta,5.2868,0.1347
